# Práctica — Clase 7: completa la ruta POST /chatbot de Flask

En el proyecto guiado real de IBM (`LLM_application_chatbot-main`), el backend Flask
(`app.py`) ya viene resuelto, pero el frontend (`static/script.js`) deja el endpoint del
chatbot sin configurar (`const url = 'www.example.com';`) como ejercicio para el estudiante.
En esta práctica vas a resolver el mismo tipo de "hueco" pero del lado del servidor: vas a
completar tú mismo la función que atiende `POST /chatbot`, y luego vas a corregir el
placeholder del frontend.

## Ejercicio 1 — Completa `handle_prompt()`

Abajo tienes el andamiaje de un servidor Flask con Blenderbot ya cargado (igual que en el
notebook de clase). Completa los `TODO` de la función `handle_prompt()` para que la ruta
`POST /chatbot` funcione de principio a fin: leer el prompt del JSON recibido, construir el
historial, tokenizar, generar la respuesta, actualizar el historial y devolver un JSON con la
respuesta.

In [ ]:
!pip install -q flask flask-cors requests

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = "facebook/blenderbot-400M-distill"
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
print("Modelo cargado.")

In [ ]:
import threading
import time
from flask import Flask, request, jsonify
from flask_cors import CORS

app = Flask(__name__)
CORS(app)

conversation_history = []

@app.route('/', methods=['GET'])
def home():
    return "Servidor Flask del chatbot activo. Usa POST /chatbot con {'prompt': '...'}"

@app.route('/chatbot', methods=['POST'])
def handle_prompt():
    # TODO 1: lee el JSON de la petición con request.get_json() y guarda
    #         el campo "prompt" en input_text
    input_text = None  # <-- reemplaza esto

    # TODO 2: arma el string de historial uniendo conversation_history con "\n"
    history = None  # <-- reemplaza esto

    # TODO 3: tokeniza history + input_text con tokenizer.encode_plus(...)
    #         (usa return_tensors="pt")
    inputs = None  # <-- reemplaza esto

    # TODO 4: genera la respuesta con model.generate(**inputs, max_new_tokens=60)
    outputs = None  # <-- reemplaza esto

    # TODO 5: decodifica outputs[0] con tokenizer.decode(..., skip_special_tokens=True)
    #         y quita espacios extra con .strip()
    response = None  # <-- reemplaza esto

    # TODO 6: agrega input_text y luego response a conversation_history

    return jsonify({"response": response})

def run_flask_app():
    app.run(port=5000, use_reloader=False)

server_thread = threading.Thread(target=run_flask_app, daemon=True)
server_thread.start()
time.sleep(3)
print("Servidor Flask corriendo en http://127.0.0.1:5000")

In [ ]:
import requests

# Esta celda solo funcionará una vez que completes los TODO de la celda anterior
resp = requests.post(
    "http://127.0.0.1:5000/chatbot",
    json={"prompt": "Hola, ¿cómo estás?"}
)
print("Status:", resp.status_code)
print("Respuesta:", resp.json())

<details>
<summary>💡 Ver solución</summary>

```python
@app.route('/chatbot', methods=['POST'])
def handle_prompt():
    data = request.get_json()
    input_text = data['prompt']

    history = "\n".join(conversation_history)

    inputs = tokenizer.encode_plus(history, input_text, return_tensors="pt")

    outputs = model.generate(**inputs, max_new_tokens=60)

    response = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    conversation_history.append(input_text)
    conversation_history.append(response)

    return jsonify({"response": response})
```
</details>

## Ejercicio 2 — Corrige el endpoint del frontend

Este es el bug/ejercicio real del proyecto guiado de IBM: en `static/script.js`, la función
que envía el mensaje del usuario al backend apunta a un endpoint de mentira:

```javascript
async function makePostRequest(msg) {
  const url = 'www.example.com';  // <-- placeholder sin resolver
  const requestBody = { prompt: msg };
  const response = await fetch(url, {
    method: 'POST',
    headers: { 'Content-Type': 'application/json' },
    body: JSON.stringify(requestBody)
  });
  // ...
}
```

**Tu tarea**: reemplaza `'www.example.com'` por la URL real de tu backend Flask. En el
proyecto guiado original, el servidor se levantaba en un Cloud IDE y su URL pública terminaba
en `/chatbot`; corriendo en tu propia máquina (o en el notebook de clase), sería
`http://127.0.0.1:5000/chatbot`.

```javascript
async function makePostRequest(msg) {
  const url = // TODO: reemplaza esta línea con la URL real de tu backend + "/chatbot"
  // ...
}
```

<details>
<summary>💡 Ver solución</summary>

```javascript
async function makePostRequest(msg) {
  const url = 'http://127.0.0.1:5000/chatbot';
  const requestBody = { prompt: msg };
  const response = await fetch(url, {
    method: 'POST',
    headers: { 'Content-Type': 'application/json' },
    body: JSON.stringify(requestBody)
  });
  // ...
}
```

Si tu servidor Flask corriera en un Cloud IDE o en un servicio como IBM Cloud Code Engine, la
URL sería la dirección pública de ese servicio terminada en `/chatbot`, no `127.0.0.1`.
</details>

Vuelve al [notebook de clase](../notebooks_clase/clase07_chatbot_web_flask.ipynb) o repasa el
[Capítulo 11 del manual](../manual/11_chatbot_web_flask.md).